In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from statsmodels.distributions.empirical_distribution import ECDF

In [30]:
ecg_pt = "/gpfs/scratch/as16583/mimic/ecg_adm.csv"
cxr_pt = "/gpfs/scratch/as16583/mimic/cxr_24_72_hr_labels.csv"
labs_pt = "/gpfs/scratch/as16583/mimic/labs_adm.csv"

ecg_df = pd.read_csv(ecg_pt)[["subject_id", "hadm_id", "ecg_path"]]
cxr_df = pd.read_csv(cxr_pt)[["subject_id", "hadm_id", "cxr_path", "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion", "No Finding"]]
labs_df = pd.read_csv(labs_pt)

In [31]:
df = pd.merge(ecg_df, cxr_df, on=["subject_id", "hadm_id"], how="inner")
df = pd.merge(df, labs_df, on=["subject_id", "hadm_id"], how="inner")
print("number of admissions: ", len(df))
print("number of subjects: ", len(df["subject_id"].unique()))

number of admissions:  11622
number of subjects:  9573


## label

We focus on the five CheXpert labels.

For each label, we do the following to create the test set:
- sample 5 query samples (ECG + labs)
- sample 125 candidate samples (CXRs): 5 positive, 120 negative

In [32]:
label_columns = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

num_query = 5
num_positive = 5
num_negative = 120

assert num_positive >= num_query # because positive samples must contain the true query samples

First, we get our query samples. We ensure that each query sample is positive for exactly _one_ label.

In [33]:
query_df = pd.DataFrame()

for label in label_columns:
    # get exclusive positive subset
    other_labels = [l for l in label_columns + ["No Finding"] if l != label]
    conditions = (df[label] == 1.0) & (df[other_labels].isin([0.0, np.nan]).all(axis=1))
    assert len(df[conditions]) >= num_query, "subset we sample from must be large enough"
    query_samples = df[conditions].sample(n=num_query, replace=False, random_state=42)

    # add to query_df
    query_samples['label_name'] = label
    query_samples['label_value'] = 1.0
    query_df = pd.concat([query_df, query_samples])

Now, we sample our candidate samples. Our candidate samples can be positive for more than one label at a time. We make sure that all query samples for each label are among the positive candidate samples.

In [34]:
candidate_df = pd.DataFrame()

for label in label_columns:

    # get samples from query_df that are positive for this label
    positive_candidates = query_df[query_df.label_name == label]
    assert len(positive_candidates) == num_query

    # remove those samples from df to sample other positive samples
    num_remaining_positives = num_positive - len(positive_candidates)
    if num_remaining_positives > 0:
        df_filtered = df[~df['hadm_id'].isin(positive_candidates.hadm_id.to_list())]
        conditions = (df_filtered[label] == 1.0) & (df_filtered["No Finding"].isin([0.0, np.nan]))
        assert len(df_filtered[conditions]) >= num_remaining_positives, "subset we sample from must be large enough"
        remaining_positives = df_filtered[conditions].sample(n=num_remaining_positives, replace=False, random_state=42)
        positive_candidates = pd.concat([positive_candidates, remaining_positives], axis=0)

    # add positives to candidate_df
    positive_candidates['label_name'] = label
    positive_candidates['label_value'] = 1.0
    candidate_df = pd.concat([candidate_df, positive_candidates])

    # get negative subset
    conditions = (df[label].isin([0.0, np.nan]))
    assert len(df[conditions]) >= num_negative, "subset we sample from must be large enough"
    negative_candidates = df[conditions].sample(n=num_negative, replace=False, random_state=42)

    # add negatives to candidate_df
    negative_candidates['label_name'] = label
    negative_candidates['label_value'] = 0.0
    candidate_df = pd.concat([candidate_df, negative_candidates])

/tmp/ipykernel_235073/2384435684.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_candidates['label_name'] = label
/tmp/ipykernel_235073/2384435684.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_candidates['label_value'] = 1.0
/tmp/ipykernel_235073/2384435684.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pand

Now, we'll split our dataset to make sure there's no overlap in subject_id between train, val, and test sets.

In [35]:
test_subject_ids = pd.concat([candidate_df['subject_id'], query_df['subject_id']]).unique()
train_val_df = df[~df['subject_id'].isin(test_subject_ids)]

print("num query_df: ", len(query_df))
print("num candidate_df: ", len(candidate_df))
print("num train_val_df: ", len(train_val_df))

num query_df:  25
num candidate_df:  625
num train_val_df:  10797


In [36]:
val_size = 800

unique_subject_ids = train_val_df["subject_id"].unique()
train_subject_ids, val_subject_ids = train_test_split(unique_subject_ids, test_size=val_size, shuffle=True, random_state=42)

train_df = train_val_df[train_val_df["subject_id"].isin(train_subject_ids)]
val_df = train_val_df[train_val_df["subject_id"].isin(val_subject_ids)]
print("len(train_df): ", len(train_df))
print("len(val_df): ", len(val_df))

len(train_df):  9832
len(val_df):  965


In [37]:
assert set(train_df["subject_id"]).isdisjoint(set(val_df["subject_id"]))
assert set(train_df["hadm_id"]).isdisjoint(set(val_df["hadm_id"]))
assert set(train_df["subject_id"]).isdisjoint(set(query_df["subject_id"]))
assert set(train_df["subject_id"]).isdisjoint(set(candidate_df["subject_id"]))
assert set(val_df["subject_id"]).isdisjoint(set(query_df["subject_id"]))
assert set(val_df["subject_id"]).isdisjoint(set(candidate_df["subject_id"]))

Now, we'll normalize the lab values into percentiles.

In [38]:
labs_columns = [c for c in train_df.columns if c not in ['subject_id', 'hadm_id', 'ecg_path', 'cxr_path', 'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion', 'No Finding']]
assert len(labs_columns) == 50, "number of lab columns must be 50"

for col in labs_columns:
    ecdf = ECDF(train_df[col])
    train_df[col + "_percentile"] = ecdf(train_df[col])
    val_df[col + "_percentile"] = ecdf(val_df[col])
    query_df[col + "_percentile"] = ecdf(query_df[col])
    candidate_df[col + "_percentile"] = ecdf(candidate_df[col])

/tmp/ipykernel_235073/2387682617.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df[col + "_percentile"] = ecdf(train_df[col])
/tmp/ipykernel_235073/2387682617.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df[col + "_percentile"] = ecdf(val_df[col])
/tmp/ipykernel_235073/2387682617.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.p

We'll calculate the mean percentiles to use in `constants.py` for when the labs are missing.

In [39]:
labs_means = {}
for col in train_df.columns:
    if col.endswith("_percentile"):
        labs_means[col] = train_df[col].mean()
labs_means

{'51221_percentile': 0.5020663667101197,
 '51265_percentile': 0.5015089239665766,
 '50912_percentile': 0.5288008394568209,
 '50971_percentile': 0.5203198726398449,
 '51222_percentile': 0.5057397144277731,
 '51301_percentile': 0.5030343062886931,
 '51249_percentile': 0.5085563690504958,
 '51279_percentile': 0.5017779989420308,
 '51250_percentile': 0.5199611404881092,
 '51248_percentile': 0.5054985596921694,
 '51277_percentile': 0.5080115364486265,
 '51006_percentile': 0.5118793534471058,
 '50983_percentile': 0.5323561132146175,
 '50902_percentile': 0.5245416322120493,
 '50882_percentile': 0.5323915643676251,
 '50868_percentile': 0.5380633350624089,
 '50931_percentile': 0.5035696942151331,
 '50960_percentile': 0.5394472438016447,
 '50893_percentile': 0.520503511060677,
 '50970_percentile': 0.5173642892208302,
 '51237_percentile': 0.5515662726978412,
 '51274_percentile': 0.5113553756816718,
 '51275_percentile': 0.5109092104557542,
 '51146_percentile': 0.5516310716539077,
 '51256_percentil

In [40]:
train_df.to_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/train.csv", index=False)
val_df.to_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/val.csv", index=False)
query_df.to_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/query.csv", index=False)
candidate_df.to_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/candidate.csv", index=False)

Now, we take our val set, and create query and candidate splits for evaluation during training.

In [11]:
val_df = pd.read_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/val.csv")

In [12]:
label_columns = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

num_query = 5
num_positive = 5
num_negative = 120

assert num_positive >= num_query # because positive samples must contain the true query samples

In [13]:
query_df = pd.DataFrame()

for label in label_columns:
    # get exclusive positive subset
    other_labels = [l for l in label_columns + ["No Finding"] if l != label]
    conditions = (val_df[label] == 1.0) & (val_df[other_labels].isin([0.0, np.nan]).all(axis=1))
    assert len(val_df[conditions]) >= num_query, "subset we sample from must be large enough"
    query_samples = val_df[conditions].sample(n=num_query, replace=False, random_state=42)

    # add to query_df
    query_samples['label_name'] = label
    query_samples['label_value'] = 1.0
    query_df = pd.concat([query_df, query_samples])

In [14]:
candidate_df = pd.DataFrame()

for label in label_columns:

    # get samples from query_df that are positive for this label
    positive_candidates = query_df[query_df.label_name == label]
    assert len(positive_candidates) == num_query

    # remove those samples from df to sample other positive samples
    num_remaining_positives = num_positive - len(positive_candidates)
    if num_remaining_positives > 0:
        df_filtered = val_df[~val_df['hadm_id'].isin(positive_candidates.hadm_id.to_list())]
        conditions = (df_filtered[label] == 1.0) & (df_filtered["No Finding"].isin([0.0, np.nan]))
        assert len(df_filtered[conditions]) >= num_remaining_positives, "subset we sample from must be large enough"
        remaining_positives = df_filtered[conditions].sample(n=num_remaining_positives, replace=False, random_state=42)
        positive_candidates = pd.concat([positive_candidates, remaining_positives], axis=0)

    # add positives to candidate_df
    positive_candidates['label_name'] = label
    positive_candidates['label_value'] = 1.0
    candidate_df = pd.concat([candidate_df, positive_candidates])

    # get negative subset
    conditions = (val_df[label].isin([0.0, np.nan]))
    assert len(val_df[conditions]) >= num_negative, "subset we sample from must be large enough"
    negative_candidates = val_df[conditions].sample(n=num_negative, replace=False, random_state=42)

    # add negatives to candidate_df
    negative_candidates['label_name'] = label
    negative_candidates['label_value'] = 0.0
    candidate_df = pd.concat([candidate_df, negative_candidates])

/tmp/ipykernel_300692/429189924.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_candidates['label_name'] = label
/tmp/ipykernel_300692/429189924.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  positive_candidates['label_value'] = 1.0
/tmp/ipykernel_300692/429189924.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-

In [15]:
query_df.to_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/query_val.csv", index=False)
candidate_df.to_csv("/gpfs/scratch/as16583/symile/src/healthcare/datasets/candidate_val.csv", index=False)